In [1]:
!apt-get update > /dev/null 2>&1
!pip install selenium > /dev/null 2>&1
!apt install chromium-chromedriver > /dev/null 2>&1
# /dev/null 2>&1 설치되는 메세지 생략 기능

In [84]:
import requests
import pandas as pd
import time
from tqdm.notebook import tqdm
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys

In [5]:
options = webdriver.ChromeOptions()
options.add_argument('--headless') # 화면 출력 x
options.add_argument('--no-sandbox') # ?
options.add_argument('--single-process')
options.add_argument('--disable-dev-shm-usage') # /deb/shm 디렉토리를 사용하지 않음 공유메모리를 담당
driver = webdriver.Chrome('chromedriver', options=options)

In [80]:
url = 'https://www.danawa.com/'
driver.get(url)
driver.find_element(By.CSS_SELECTOR, '.search__input').send_keys('무선청소기')
driver.find_element(By.CSS_SELECTOR, '.search__input').send_keys(Keys.ENTER)
soup = BeautifulSoup(driver.page_source, 'html.parser')

Step.1 무선청소기에 대한 정보 가져오기

In [66]:
lis = soup.select('li.prod_item')

In [67]:
len(lis)

44

In [68]:
# 제품명
name = soup.select_one('.prod_name > a').get_text()
name

'LG전자 코드제로 A9S AT9270'

In [49]:
# 스펙
spec = soup.select_one('.spec_list').get_text().replace('\t', '').replace('\n', '')
spec

'핸디스틱청소기 / 무선형 / 흡입전용 / 흡입력: 210W / [구성] 바닥 / 솔형 / 틈새 / 연장툴 / 청정스테이션 / [배터리] 분리형 (1개) / 충전시간: 3시간30분 / 사용시간(개당): 1시간 / [성능] 디지털인버터모터 / 5단계여과 / [기능] 물걸레(별도구매) / 디스플레이표시창 / 자동물공급 / 배터리잔량표시 / 자동먼지비움 / [부가] 먼지통용량: 0.5L / 물통용량: 0.15L / 색상: 미드나잇블루 / 무게: 2.5kg / 액세서리크래들  / 크기(가로x세로x깊이): 250x930x202mm'

In [63]:
# 가격
price = soup.select_one('.price_sect > a > strong').get_text().replace(',', '')
price

'458920'

Step.2 1페이지 데이터 추출해보기

In [81]:
# 상품 정보 태그에서 원하는 정보를 추출하는 함수를 만들기
def get_prod_items(lis):
  prod_data = []

  for prod_item in lis:
    # 공백 ('li.prod_item.product-pot') 제외
    if 'product-pot' in prod_item['class']:
      continue
    try:
      name = lis[0].select('.prod_name > a')[0].get_text()
      spec = lis[0].select('.spec_list')[0].get_text().replace('\t', '').replace('\n', '')
      price = lis[0].select('.price_sect > a > strong')[0].get_text().replace(',', '')
      prod_data.append([name, spec, price])
    except:
      pass
    return prod_data

In [82]:
# 상품 정보를 가져오는 함수 테스트
lis = soup.select('li.prod_item')
prod_data = get_prod_items(lis)
print(prod_data)

[['삼성전자 비스포크 제트 VS20A956A3', '핸디스틱청소기 / 무선형 / 흡입전용 / 흡입력: 210W / [구성] 바닥 / 솔형 / 틈새 / 연장툴 / 청정스테이션 / [배터리] 분리형 (1개) / 충전시간: 3시간30분 / 사용시간(개당): 1시간 / [성능] 디지털인버터모터 / 5단계여과 / [기능] 물걸레(별도구매) / 디스플레이표시창 / 자동물공급 / 배터리잔량표시 / 자동먼지비움 / [부가] 먼지통용량: 0.5L / 물통용량: 0.15L / 색상: 미드나잇블루 / 무게: 2.5kg / 액세서리크래들  / 크기(가로x세로x깊이): 250x930x202mm', '458920']]


Step.3 여러페이지에 걸친 다나와 검색 페이지 크롤링

In [83]:
# 다나와 검색 URL을 만들어 주는 함수
def get_search_page_url(keyword, page):
  return f'https://search.danawa.com/dsearch.php?query={keyword}&originalQuery={keyword}&previousKeyword={keyword}&volumeType=allvs&page={page}&limit=40&sort=saveDESC&list=list&boost=true&addDelivery=N&recommendedSort=Y&defaultUICategoryCode=10325109&defaultPhysicsCategoryCode=72%7C80%7C81%7C0&defaultVmTab=2942&defaultVaTab=719721&tab=main'

keyword = '무선청소기'
page = 3
url = get_search_page_url(keyword, page)
print(url) 

https://search.danawa.com/dsearch.php?query=무선청소기&originalQuery=무선청소기&previousKeyword=무선청소기&volumeType=allvs&page=3&limit=40&sort=saveDESC&list=list&boost=true&addDelivery=N&recommendedSort=Y&defaultUICategoryCode=10325109&defaultPhysicsCategoryCode=72%7C80%7C81%7C0&defaultVmTab=2942&defaultVaTab=719721&tab=main


In [89]:
keyword = '무선청소기'
total_page = 10
prod_data_total = []
for page in tqdm(range(1, total_page+1)):
  url = get_search_page_url(keyword, page)
  driver.get(url)
  time.sleep(5)

  html = driver.page_source
  soup = BeautifulSoup(html, 'html.parser')

  # 상품정보 추출
  lis = soup.select('li.prod_item')
  prod_item_list = get_prod_items(lis)

  # 추출한 데이터 저장
  prod_data_total = prod_data_total + prod_item_list

  0%|          | 0/10 [00:00<?, ?it/s]

Step.4 수집한 데이터 저장

In [90]:
df = pd.DataFrame(prod_data_total)
df.columns = ['상품명', '스펙 목록', '가격']

In [ ]:
df

In [92]:
df.to_csv('danawa_crawling.csv', index = False)